In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"])

In [0]:
import sys
sys.path.append("../src")
from upi.config import get_config

cfg = get_config(dbutils.widgets.get("env"))
print(f"env={cfg.env}  catalog={cfg.catalog}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg.catalog}")

for schema in ["bronze", "silver", "gold", "ops", "quarantine"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cfg.schema(schema)}")

display(spark.sql(f"SHOW SCHEMAS IN {cfg.catalog}"))

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.schema('bronze')}.landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {cfg.schema('bronze')}.checkpoints")

for folder in ["npci", "transactions", "status_updates", "dims", "disputes",
               "settlement", "stream_sample", "stream", "_held"]:
    dbutils.fs.mkdirs(cfg.raw(folder))

display(dbutils.fs.ls(cfg.landing))


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {cfg.table('ops', 'cdc_watermarks')} (
        source_table    STRING,
        high_water_mark TIMESTAMP,
        updated_at      TIMESTAMP
    )
""")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {cfg.table('ops', 'cdc_runs')} (
        run_id STRING, source_table STRING, started_at TIMESTAMP, finished_at TIMESTAMP,
        from_mark TIMESTAMP, to_mark TIMESTAMP, rows_read BIGINT, status STRING, error STRING
    )
""")

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {cfg.table('ops', 'dq_metrics')} (
        dataset STRING, expectation STRING, severity STRING,
        rows_checked BIGINT, rows_failed BIGINT, failure_rate DOUBLE, measured_at TIMESTAMP
    )
""")

display(spark.sql(f"SHOW TABLES IN {cfg.schema('ops')}"))

In [0]:
GRANTS_ENABLED = False

if GRANTS_ENABLED:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {cfg.catalog} TO `analysts`")
    for schema in ["silver", "gold"]:
        spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {cfg.schema(schema)} TO `analysts`")
    print("grants applied")
else:
    print("grants skipped — create the analysts group first")
